<a href="https://colab.research.google.com/github/Uday-Naik-coder/Explainable-Misinformation-Detection-using-DANN-and-LIME-models/blob/main/Explainable_Misinformation_Detection_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install -q sentence-transformers lime tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import os, math, random
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from lime.lime_text import LimeTextExplainer
from torch.autograd import Function
from sklearn.model_selection import train_test_split

# Path to your dataset (upload misovac.csv into /content)
CSV_PATH = "/content/misovac_dataset.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_DIR = "/content/misovac_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

print("DEVICE:", DEVICE)

DEVICE: cuda


In [3]:
def load_misovac(path):
    df = pd.read_csv(path)

    df.columns = [c.lower().strip() for c in df.columns]

    if "text" not in df.columns:
        raise ValueError("Dataset must include a 'text' column")
    if "label" not in df.columns:
        raise ValueError("Dataset must include a 'label' column")

    # Normalize labels
    # Create a temporary column for normalized labels, then replace the original 'label'
    normalized_labels = df["label"].astype(str).str.lower().str.strip()
    label_map = {
        "fake":"fake","false":"fake","0":"fake","f":"fake",
        "real":"real","true":"real","1":"real","r":"real"
    }
    df["label"] = normalized_labels.map(label_map).fillna(normalized_labels) # Overwrite the original 'label' column

    # Domain / platform column
    if "platform" not in df.columns:
        df["platform"] = "misovac"

    # Select only the desired columns, ensuring no duplicates.
    # The 'label' column now contains the normalized string labels.
    df = df[["text","label","platform"]].dropna()

    return df

df = load_misovac(CSV_PATH)
print("Loaded rows:", len(df))
df.head()

Loaded rows: 1000


,text,label,platform
0,Following public health guidelines helps prote...,real,Twitter
1,Government is using vaccines to implant tracki...,fake,Twitter
2,Research demonstrates vaccine effectiveness in...,real,Twitter
3,Masks contain toxic chemicals that cause lung ...,fake,Twitter
4,Studies show masks reduce transmission by bloc...,real,Twitter


In [4]:
le_label = LabelEncoder()
df["label_enc"] = le_label.fit_transform(df["label"])

le_domain = LabelEncoder()
df["domain_enc"] = le_domain.fit_transform(df["platform"])

print("Label mapping:", dict(zip(le_label.classes_, le_label.transform(le_label.classes_))))
print("Domain mapping:", dict(zip(le_domain.classes_, le_domain.transform(le_domain.classes_))))

Label mapping: {'fake': np.int64(0), 'real': np.int64(1)}
Domain mapping: {'Instagram': np.int64(0), 'Reddit': np.int64(1), 'Twitter': np.int64(2), 'YouTube': np.int64(3)}


In [5]:
print(df.columns)
df.head(3)

Index(['text', 'label', 'platform', 'label_enc', 'domain_enc'], dtype='object')


,text,label,platform,label_enc,domain_enc
0,Following public health guidelines helps prote...,real,Twitter,1,2
1,Government is using vaccines to implant tracki...,fake,Twitter,0,2
2,Research demonstrates vaccine effectiveness in...,real,Twitter,1,2


In [6]:
sbert = SentenceTransformer("all-MiniLM-L6-v2")

EMB_FILE = os.path.join(CACHE_DIR, "emb.npy")
META_FILE = os.path.join(CACHE_DIR, "meta.csv")

if os.path.exists(EMB_FILE) and os.path.exists(META_FILE):
    meta = pd.read_csv(META_FILE)
    if len(meta) == len(df):
        embeddings = np.load(EMB_FILE)
        print("Loaded cached embeddings.")
    else:
        raise ValueError("Cache mismatch. Delete cache folder and rerun.")
else:
    print("Computing embeddings...")
    texts = df["text"].tolist()
    batch_size = 64
    emb_list = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        b_emb = sbert.encode(batch, convert_to_numpy=True, show_progress_bar=False)
        emb_list.append(b_emb)
    embeddings = np.vstack(emb_list)
    np.save(EMB_FILE, embeddings)
    df.to_csv(META_FILE, index=False)

print("Embeddings:", embeddings.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Computing embeddings...


  0%|          | 0/16 [00:00<?, ?it/s]

Embeddings: (1000, 384)


In [7]:
class EmbDataset(Dataset):
    def __init__(self, X, y, d, idx):
        self.X = X[idx]
        self.y = y[idx]
        self.d = d[idx]
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return (
            torch.tensor(self.X[i], dtype=torch.float32),
            torch.tensor(self.y[i], dtype=torch.long),
            torch.tensor(self.d[i], dtype=torch.long),
        )

indices = np.arange(len(df))
strat = np.stack([df["label_enc"], df["domain_enc"]], axis=1)

train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42, stratify=strat)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42,
    stratify=np.stack([df.loc[temp_idx,"label_enc"], df.loc[temp_idx,"domain_enc"]], axis=1))

train_ds = EmbDataset(embeddings, df["label_enc"].values, df["domain_enc"].values, train_idx)
val_ds   = EmbDataset(embeddings, df["label_enc"].values, df["domain_enc"].values, val_idx)
test_ds  = EmbDataset(embeddings, df["label_enc"].values, df["domain_enc"].values, test_idx)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64)
test_loader  = DataLoader(test_ds, batch_size=64)

print("Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))


Train: 700 Val: 150 Test: 150


In [8]:
class GRL(Function):
    @staticmethod
    def forward(ctx, x, coeff):
        ctx.coeff = coeff
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad):
        return -ctx.coeff * grad, None

def grad_reverse(x, c):
    return GRL.apply(x, c)

class DANN(nn.Module):
    def __init__(self, dim=384, hidden=256, n_labels=2, n_domains=2):
        super().__init__()
        self.feat = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.label_clf = nn.Sequential(
            nn.Linear(hidden, hidden//2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden//2, n_labels)
        )
        self.domain_clf = nn.Sequential(
            nn.Linear(hidden, hidden//2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden//2, n_domains)
        )

    def forward(self, x, coeff=1.0):
        f = self.feat(x)
        lbl = self.label_clf(f)
        f_rev = grad_reverse(f, coeff)
        dom = self.domain_clf(f_rev)
        return lbl, dom

model = DANN(
    dim=embeddings.shape[1],
    n_labels=len(le_label.classes_),
    n_domains=len(le_domain.classes_)
).to(DEVICE)

crit_lbl = nn.CrossEntropyLoss()
crit_dom = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(model.parameters(), lr=2e-4)


In [9]:
def grl_coeff(step, total, alpha=10):
    p = step / total
    return 2/(1+math.exp(-alpha*p)) - 1

EPOCHS = 6
total_steps = EPOCHS * len(train_loader)
global_step = 0
best_f1 = 0
BEST_PATH = os.path.join(CACHE_DIR, "best_dann.pt")

def evaluate(loader):
    model.eval()
    yp, yt = [], []
    with torch.no_grad():
        for xb, yb, db in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            out,_ = model(xb, 0)
            pred = torch.argmax(out,1)
            yp.extend(pred.cpu().numpy())
            yt.extend(yb.cpu().numpy())
    return f1_score(yt, yp, average="macro")

for epoch in range(EPOCHS):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for xb, yb, db in pbar:
        xb, yb, db = xb.to(DEVICE), yb.to(DEVICE), db.to(DEVICE)

        opt.zero_grad()
        coeff = grl_coeff(global_step, total_steps)

        out_lbl, out_dom = model(xb, coeff)
        loss = crit_lbl(out_lbl, yb) + 0.5 * crit_dom(out_dom, db)

        loss.backward()
        opt.step()
        global_step += 1
        pbar.set_postfix({"loss": loss.item()})

    val_f1 = evaluate(val_loader)
    print("VAL F1:", val_f1)
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), BEST_PATH)
        print("Saved best model.")


Epoch 1/6:   0%|          | 0/11 [00:00<?, ?it/s]

VAL F1: 0.3333333333333333
Saved best model.


Epoch 2/6:   0%|          | 0/11 [00:00<?, ?it/s]

VAL F1: 0.3333333333333333


Epoch 3/6:   0%|          | 0/11 [00:00<?, ?it/s]

...

VAL F1: 1.0
Saved best model.


Epoch 5/6:   0%|          | 0/11 [00:00<?, ?it/s]

VAL F1: 1.0


Epoch 6/6:   0%|          | 0/11 [00:00<?, ?it/s]

VAL F1: 1.0


In [10]:
model.load_state_dict(torch.load(BEST_PATH))
model.eval()

yt, yp = [], []
with torch.no_grad():
    for xb, yb, db in test_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        out,_ = model(xb, 0)
        pred = torch.argmax(out,1)
        yp.extend(pred.cpu().numpy())
        yt.extend(yb.cpu().numpy())

print(classification_report(yt, yp, target_names=list(le_label.classes_)))
print(confusion_matrix(yt, yp))


              precision    recall  f1-score   support

        fake       1.00      1.00      1.00        75
        real       1.00      1.00      1.00        75

    accuracy                           1.00       150
   macro avg       1.00      1.00      1.00       150
weighted avg       1.00      1.00      1.00       150

[[75  0]
 [ 0 75]]


In [11]:
def predict_proba(texts):
    em = sbert.encode(texts, convert_to_numpy=True)
    xb = torch.tensor(em, dtype=torch.float32).to(DEVICE)
    out,_ = model(xb,0)
    return torch.softmax(out, dim=1).detach().cpu().numpy()

explainer = LimeTextExplainer(class_names=list(le_label.classes_))

sample_texts = df.loc[test_idx[:4],"text"].tolist()

for t in sample_texts:
    exp = explainer.explain_instance(t, predict_proba, num_samples=500, num_features=6)
    print("TEXT:", t[:150], "...")
    print(exp.as_list())
    print("-"*50)

TEXT: Vaccines shed and infect unvaccinated people nearby. @WHO ...
[(np.str_('unvaccinated'), -0.012197302389262308), (np.str_('nearby'), 0.007706725283981063), (np.str_('infect'), -0.006559145141546057), (np.str_('Vaccines'), -0.0046438098814460035), (np.str_('people'), 0.003908911259853675), (np.str_('WHO'), -0.0031323790279004565)]
--------------------------------------------------
TEXT: Healthcare workers recommend following evidence-based prevention strategies. Edit: ...
[(np.str_('prevention'), 0.016898533759594284), (np.str_('strategies'), 0.00993614802773842), (np.str_('workers'), 0.008886338516949043), (np.str_('evidence'), 0.008841672003370741), (np.str_('following'), 0.005687595855843597), (np.str_('recommend'), 0.004510485490517156)]
--------------------------------------------------
TEXT: Social distancing remains an important preventive measure in crowded areas. @WHO ...
[(np.str_('preventive'), 0.016656610962226256), (np.str_('crowded'), 0.012390241814863411), (np.str_(

In [12]:
# Make sure you've already run previous cells: embeddings computed, model trained and saved, sbert and df exist.

import json
from IPython.display import display, Markdown

# Load best model if not in memory
if not any(True for _ in model.parameters()):
    model.load_state_dict(torch.load(BEST_PATH))
model.eval()

# Map integer labels back to text
label_names = list(le_label.classes_)   # e.g., ['fake','real']
domain_names = list(le_domain.classes_)

# ---- A) CUSTOM TEXTS ----
# Replace these with any safe examples you'd like to test.
# DO NOT add novel false medical claims. Use neutral or dataset-provided lines.
custom_texts = [
    "Vaccination reduces the risk of severe disease and hospitalization from COVID-19.",
    "I heard drinking hot water cures COVID-19 overnight, share if true!",
    "New policy announced by the government will affect travel rules for international students.",
    "Studies show that mask wearing helps reduce spread of respiratory diseases."
]

print("Running predictions on custom texts...\n")
with torch.no_grad():
    embs = sbert.encode(custom_texts, convert_to_numpy=True)
    xb = torch.tensor(embs, dtype=torch.float32).to(DEVICE)
    logits, _ = model(xb, coeff=0.0)
    probs = torch.softmax(logits, dim=1).cpu().numpy()
for i, txt in enumerate(custom_texts):
    pred_idx = int(probs[i].argmax())
    pred_label = label_names[pred_idx]
    pred_prob = float(probs[i][pred_idx])
    print(f"Example {i+1}:")
    display(Markdown(f"**Text:** {txt}"))
    print(f"Predicted: {pred_label} (prob={pred_prob:.3f}) — probs: {probs[i].tolist()}")
    print("-"*80)

# ---- B) A FEW SAMPLES FROM THE TEST SET (already indexed as test_idx) ----
print("\nRunning predictions on a few test-set samples...\n")
n_show = min(6, len(test_idx))
sample_test_indices = list(test_idx[:n_show])

test_texts = df.loc[sample_test_indices, "text"].astype(str).tolist()
true_labels = df.loc[sample_test_indices, "label"].tolist()
test_platforms = df.loc[sample_test_indices, "platform"].tolist()

with torch.no_grad():
    embs_test = sbert.encode(test_texts, convert_to_numpy=True)
    xb_test = torch.tensor(embs_test, dtype=torch.float32).to(DEVICE)
    logits_test, _ = model(xb_test, coeff=0.0)
    probs_test = torch.softmax(logits_test, dim=1).cpu().numpy()

for i, idx in enumerate(sample_test_indices):
    txt = test_texts[i]
    true_lab = true_labels[i]
    plat = test_platforms[i]
    p = probs_test[i]
    pred_idx = int(p.argmax())
    pred_label = label_names[pred_idx]
    pred_prob = float(p[pred_idx])
    print(f"Test sample idx={idx}  platform={plat}")
    display(Markdown(f"**Text:** {txt[:400]}{'...' if len(txt)>400 else ''}"))
    print(f"True label: {true_lab}  | Predicted: {pred_label} (prob={pred_prob:.3f})  | Probs: {p.tolist()}")
    print("-"*80)

# ---- Optional: save predictions to CSV ----
out_pred = []
for i, idx in enumerate(sample_test_indices):
    out_pred.append({
        "index": int(idx),
        "platform": test_platforms[i],
        "text": test_texts[i],
        "true_label": true_labels[i],
        "pred_label": label_names[int(probs_test[i].argmax())],
        "pred_probs": json.dumps(probs_test[i].tolist())
    })
out_df = pd.DataFrame(out_pred)
out_csv = os.path.join(CACHE_DIR, "sample_test_predictions.csv")
out_df.to_csv(out_csv, index=False)
print("Saved sample test predictions to:", out_csv)

Running predictions on custom texts...

Example 1:


**Text:** Vaccination reduces the risk of severe disease and hospitalization from COVID-19.

Predicted: real (prob=0.525) — probs: [0.4754260182380676, 0.5245739817619324]
--------------------------------------------------------------------------------
Example 2:


**Text:** I heard drinking hot water cures COVID-19 overnight, share if true!

Predicted: fake (prob=0.504) — probs: [0.5041832327842712, 0.495816707611084]
--------------------------------------------------------------------------------
Example 3:


...

True label: real  | Predicted: real (prob=0.532)  | Probs: [0.46782955527305603, 0.5321704149246216]
--------------------------------------------------------------------------------
Test sample idx=786  platform=Reddit


**Text:** Scientific consensus supports vaccination as the best defense against COVID-19. r/COVID19

True label: real  | Predicted: real (prob=0.548)  | Probs: [0.4519059360027313, 0.5480940341949463]
--------------------------------------------------------------------------------
Test sample idx=170  platform=Twitter


**Text:** Research demonstrates vaccine effectiveness in preventing hospitalization. @WHO

True label: real  | Predicted: real (prob=0.553)  | Probs: [0.44722843170166016, 0.5527715682983398]
--------------------------------------------------------------------------------
Saved sample test predictions to: /content/misovac_cache/sample_test_predictions.csv


In [13]:
# WARNING: LIME is moderately slow. Use few examples (<=4) and num_samples=300 to keep it quick.
from lime.lime_text import LimeTextExplainer

explainer = LimeTextExplainer(class_names=label_names)

# Choose examples to explain: combine some custom and some test samples
example_texts = custom_texts[:2] + test_texts[:2]  # adjust as needed (keep small)
print("Explaining {} examples with LIME (num_samples=300)".format(len(example_texts)))

def predict_proba_for_lime(texts):
    embs = sbert.encode(texts, convert_to_numpy=True)
    xb = torch.tensor(embs, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        out_lbl, _ = model(xb, coeff=0.0)
        probs = torch.softmax(out_lbl, dim=1).cpu().numpy()
    return probs

for txt in example_texts:
    print("="*80)
    display(Markdown(f"**TEXT:** {txt}"))
    exp = explainer.explain_instance(txt, predict_proba_for_lime, num_features=6, num_samples=300)
    print("LIME explanation (feature, weight):")
    for feat, weight in exp.as_list():
        print(f"  {feat} -> {weight:.3f}")
    print("\n")

Explaining 4 examples with LIME (num_samples=300)


**TEXT:** Vaccination reduces the risk of severe disease and hospitalization from COVID-19.

LIME explanation (feature, weight):
  Vaccination -> -0.009
  risk -> 0.009
  hospitalization -> 0.009
  19 -> 0.006
  reduces -> 0.006
  COVID -> -0.003




**TEXT:** I heard drinking hot water cures COVID-19 overnight, share if true!

LIME explanation (feature, weight):
  COVID -> -0.011
  cures -> -0.007
  share -> 0.006
  true -> -0.004
  if -> 0.002
  hot -> 0.002




**TEXT:** Vaccines shed and infect unvaccinated people nearby. @WHO

LIME explanation (feature, weight):
  unvaccinated -> -0.013
  nearby -> 0.008
  infect -> -0.006
  WHO -> -0.003
  people -> 0.003
  Vaccines -> -0.003




**TEXT:** Healthcare workers recommend following evidence-based prevention strategies. Edit:

LIME explanation (feature, weight):
  prevention -> 0.017
  strategies -> 0.011
  evidence -> 0.009
  workers -> 0.008
  recommend -> 0.007
  following -> 0.006




In [14]:
import os, joblib, numpy as np, pandas as pd, torch

# Where to save inside Google Drive
SAVE_DIR = "/content/drive/MyDrive/misinformation_model"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Saving to:", SAVE_DIR)

# Paths created during your training (update ONLY if different):
LOCAL_MODEL_PATH = "/content/misovac_cache/dann_best.pt"
LOCAL_EMB_PATH   = "/content/misovac_cache/emb.npy"
LOCAL_META_CSV   = "/content/misovac_cache/meta.csv"   # contains cleaned dataset
LOCAL_TEST_PRED  = "/content/misovac_cache/test_predictions.csv"  # optional

# ---------- Save model ----------
if os.path.exists(LOCAL_MODEL_PATH):
    !cp "{LOCAL_MODEL_PATH}" "{SAVE_DIR}/dann_best.pt"
    print("✔ Model saved.")
else:
    print("Model NOT found:", LOCAL_MODEL_PATH)

# ---------- Save embeddings ----------
if os.path.exists(LOCAL_EMB_PATH):
    !cp "{LOCAL_EMB_PATH}" "{SAVE_DIR}/embeddings.npy"
    print("✔ Embeddings saved.")
else:
    print("Embeddings NOT found:", LOCAL_EMB_PATH)

# ---------- Save cleaned dataframe ----------
if os.path.exists(LOCAL_META_CSV):
    !cp "{LOCAL_META_CSV}" "{SAVE_DIR}/data_cleaned.csv"
    print("✔ Cleaned dataset saved.")
else:
    try:
        df.to_csv(f"{SAVE_DIR}/data_cleaned.csv", index=False)
        print("✔ df saved from memory.")
    except:
        print("❌ No df or meta CSV found.")

# ---------- Save label/domain encoders ----------
try:
    joblib.dump(le_label, f"{SAVE_DIR}/label_encoder.pkl")
    joblib.dump(le_domain, f"{SAVE_DIR}/domain_encoder.pkl")
    print("✔ Encoders saved.")
except:
    print("❌ Encoders not found. Make sure le_label and le_domain exist.")

# ---------- Save test predictions (if exist) ----------
if os.path.exists(LOCAL_TEST_PRED):
    !cp "{LOCAL_TEST_PRED}" "{SAVE_DIR}/test_predictions.csv"
    print("✔ Test predictions saved.")
else:
    print("No test predictions found (optional).")

print("\n📁 Saved files:")
!ls -la "{SAVE_DIR}"


Saving to: /content/drive/MyDrive/misinformation_model
Model NOT found: /content/misovac_cache/dann_best.pt
✔ Embeddings saved.
✔ Cleaned dataset saved.
✔ Encoders saved.
No test predictions found (optional).

📁 Saved files:
total 1612
drwxr-xr-x 2 root root    4096 Apr  3 04:39 .
drwxr-xr-x 3 root root    4096 Apr  3 04:39 ..
-rw-r--r-- 1 root root   92978 Apr  3 04:39 data_cleaned.csv
-rw-r--r-- 1 root root     514 Apr  3 04:39 domain_encoder.pkl
-rw-r--r-- 1 root root 1536128 Apr  3 04:39 embeddings.npy
-rw-r--r-- 1 root root     487 Apr  3 04:39 label_encoder.pkl


In [15]:
import random
from IPython.display import Markdown, display

def test_random_samples(n=5):
    """Test model predictions on n random dataset samples."""
    samples = df.sample(n)
    texts = samples["text"].tolist()
    true_labels = samples["label"].tolist()

    probs = predict_proba(texts)

    for i, t in enumerate(texts):
        pred_idx = probs[i].argmax()
        pred_label = label_names[pred_idx]
        pred_prob = probs[i][pred_idx]

        display(Markdown(f"### 🔍 Sample {i+1}"))
        display(Markdown(f"**Text:** {t}"))
        print(f"True Label: {true_labels[i]}")
        print(f"Predicted Label: {pred_label}")
        print(f"Confidence: {pred_prob:.4f}")
        print("-"*80)

print("Testing random samples...\n")
test_random_samples(5)


Testing random samples...



### 🔍 Sample 1

**Text:** Studies show masks reduce transmission by blocking respiratory droplets. Subscribe

True Label: real
Predicted Label: real
Confidence: 0.5383
--------------------------------------------------------------------------------


### 🔍 Sample 2

...

**Text:** Testing helps identify cases early and prevent further spread. 😷

True Label: real
Predicted Label: real
Confidence: 0.5874
--------------------------------------------------------------------------------


### 🔍 Sample 5

**Text:** CDC recommends vaccination for all eligible individuals to prevent severe COVID-19. #COVID19

True Label: real
Predicted Label: real
Confidence: 0.5502
--------------------------------------------------------------------------------


In [16]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate_full_testset():
    yt, yp = [], []
    with torch.no_grad():
        for xb, yb, db in test_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            out,_ = model(xb,0)
            preds = torch.argmax(out,1)
            yt.extend(yb.cpu().numpy())
            yp.extend(preds.cpu().numpy())

    print("Accuracy:", accuracy_score(yt, yp))
    print("Macro F1:", f1_score(yt, yp, average="macro"))

print("Full Test Set Evaluation:")
evaluate_full_testset()


Full Test Set Evaluation:
Accuracy: 1.0
Macro F1: 1.0


In [17]:
def check_consistency(text, repeat=5):
    print(f"Checking consistency for:\n{text}\n")
    for i in range(repeat):
        probs = predict_proba([text])[0]
        print(f"Run {i+1}: Pred={label_names[probs.argmax()]}  Prob={probs.max():.4f}")

check_consistency("Vaccines reduce the chance of severe COVID-19 infection.")


Checking consistency for:
Vaccines reduce the chance of severe COVID-19 infection.

Run 1: Pred=real  Prob=0.5088
Run 2: Pred=real  Prob=0.5088
Run 3: Pred=real  Prob=0.5088
Run 4: Pred=real  Prob=0.5088
Run 5: Pred=real  Prob=0.5088


In [18]:
def explain_text(text):
    print("="*90)
    display(Markdown("### 🧠 LIME Explanation"))
    display(Markdown(f"**Text:** {text}"))

    exp = explainer.explain_instance(
        text,
        predict_proba_for_lime,
        num_features=8,
        num_samples=400
    )

    print("\nTop words influencing the prediction:")
    for word, weight in exp.as_list():
        print(f"{word}: {weight:.3f}")

    return exp

# Example usage
explain_text("I read online that drinking hot water can cure COVID instantly.")
explain_text("Vaccines significantly reduce the risk of hospitalization and severe illness.")

### 🧠 LIME Explanation

**Text:** I read online that drinking hot water can cure COVID instantly.


Top words influencing the prediction:
cure: -0.011
COVID: -0.007
instantly: 0.005
read: 0.003
I: -0.003
water: -0.003
drinking: 0.003
online: -0.001


### 🧠 LIME Explanation

**Text:** Vaccines significantly reduce the risk of hospitalization and severe illness.


Top words influencing the prediction:
Vaccines: -0.034
hospitalization: 0.011
significantly: 0.009
reduce: 0.007
risk: 0.007
severe: -0.002
and: -0.001
of: 0.001


In [19]:
def model_diagnostic(text):
    print("="*120)
    display(Markdown("## 🔍 Full Model Diagnostic"))
    display(Markdown(f"**Input Text:** {text}"))

    # Prediction
    probs = predict_proba([text])[0]
    pred_idx = probs.argmax()
    pred_label = label_names[pred_idx]

    print(f"\nPredicted Label: {pred_label}")
    print(f"Confidence: {probs[pred_idx]:.4f}")
    print(f"Full Probabilities: {probs}")

    # Explanation
    exp = explainer.explain_instance(
        text,
        predict_proba_for_lime,
        num_features=6,
        num_samples=300
    )

    print("\n🧠 LIME Explanation (word -> weight):")
    for feat, w in exp.as_list():
        print(f"  {feat}: {w:.3f}")

    print("="*120)
extra_test_texts = [
    "Vaccines significantly reduce the risk of hospitalization and severe illness.",
    "Wearing masks helps reduce the spread of respiratory infections.",
    "COVID-19 spreads mainly through respiratory droplets.",
    "Boiling water before drinking removes most bacteria and parasites.",
    "Exercise and a balanced diet support overall immunity.",
    "The Earth revolves around the Sun once every 365 days.",
    "Seatbelts reduce the risk of injury in vehicle accidents.",
    "Handwashing with soap is effective in removing viruses.",
    "Modern aircraft rely on both autopilot systems and human pilots.",
    "Antibiotics treat bacterial infections, not viral infections.",
    "Regular physical activity lowers the risk of heart disease.",
    "Climate change is influenced by human activities like burning fossil fuels.",
    "Vaccines must pass multiple safety evaluations before approval.",
    "Most smartphones receive security updates for several years after release.",
    "Drinking water is essential for maintaining healthy bodily functions.",
    "Public transportation reduces overall traffic congestion.",
    "Renewable energy sources such as solar power help reduce carbon emissions.",
    "Machine learning models improve through exposure to data.",
    "Professional journalists verify information before publishing news.",
    "High blood pressure increases the risk of stroke and heart disease.",

    "Drinking hot water can cure COVID-19 instantly.",
    "The virus cannot survive at temperatures above 25 degrees Celsius.",
    "Eating garlic guarantees complete immunity against viruses.",
    "A secret government chip is hidden inside every vaccine dose.",
    "COVID-19 can be cured by exposing yourself to loud noises.",
    "Holding your breath for 20 seconds can diagnose coronavirus.",
    "Rubbing lemon on your skin prevents all infections.",
    "5G towers spread COVID-19 through radio waves.",
    "The moon landing was filmed entirely in a Hollywood studio.",
    "Chocolate milk comes from brown cows.",
    "Vaccines rewrite human DNA permanently.",
    "Drinking alcohol prevents viral infections of any kind.",
    "Mobile phones attract harmful cosmic radiation during thunderstorms.",
    "COVID-19 disappears if you drink warm saltwater every morning.",
    "Magnets stick to your arm after receiving a vaccine.",
    "A simple homemade device can generate free electricity forever.",
    "Eating only sugar for a week will flush toxins from your body.",
    "If you sleep with onions in your socks, you won’t get the flu.",
    "COVID-19 was created to control people through microchip implants.",
    "Sleeping under a full moon can cure all respiratory diseases."
]


# Example:
model_diagnostic("The COVID vaccine contains microchips that track people.")
model_diagnostic("Vaccines significantly reduce the risk of hospitalization and severe illness.")
model_diagnostic("Vaccines rewrite human DNA permanently.")
model_diagnostic("Exercise and a balanced diet support overall immunity.")




## 🔍 Full Model Diagnostic

**Input Text:** The COVID vaccine contains microchips that track people.


Predicted Label: fake
Confidence: 0.5117
Full Probabilities: [0.5117322  0.48826784]

🧠 LIME Explanation (word -> weight):
  microchips: -0.016
  vaccine: -0.016
  track: 0.014
  people: 0.006
  contains: -0.003
  that: 0.002


## 🔍 Full Model Diagnostic

**Input Text:** Vaccines significantly reduce the risk of hospitalization and severe illness.

...

**Input Text:** Vaccines rewrite human DNA permanently.


Predicted Label: fake
Confidence: 0.5202
Full Probabilities: [0.5202228  0.47977725]

🧠 LIME Explanation (word -> weight):
  Vaccines: -0.017
  DNA: -0.006
  rewrite: -0.004
  human: 0.002
  permanently: 0.001


## 🔍 Full Model Diagnostic

**Input Text:** Exercise and a balanced diet support overall immunity.


Predicted Label: real
Confidence: 0.5160
Full Probabilities: [0.48401707 0.5159829 ]

🧠 LIME Explanation (word -> weight):
  diet: -0.015
  immunity: -0.014
  Exercise: 0.011
  support: 0.007
  balanced: 0.007
  a: -0.004
